# 02 — Clean TF-IDF abstract retrieval baseline
This follows the official SciFact abstract-retrieval implementation: fit a TF-IDF vectorizer on each corpus title concatenated with its abstract sentences, use English stop words and word n-grams 1–2, score each claim against all abstracts, and rank by descending dot product. The default top-k is 3. This notebook does not reproduce the rationale selector or label prediction stage.

In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
BASE=Path("/content/scifact_study")
manifest=json.loads((BASE/"dataset_manifest.json").read_text())
def read_jsonl(path):
    with open(path,encoding="utf-8") as f: return [json.loads(x) for x in f if x.strip()]
corpus=read_jsonl(manifest["corpus_path"])
dev=read_jsonl(manifest["dev_path"])


In [ ]:
# Match the official corpus text construction and TF-IDF settings
corpus_texts=[doc["title"] + " ".join(doc["abstract"]) for doc in corpus]
vectorizer=TfidfVectorizer(stop_words="english", ngram_range=(1,2))
doc_matrix=vectorizer.fit_transform(corpus_texts)
doc_ids=np.asarray([int(d["doc_id"]) for d in corpus])
print("TF-IDF matrix:", doc_matrix.shape)


In [ ]:
# Score all dev claims; keep a sufficiently long ranking for later controlled perturbation
TOP_K=3
RESERVE_K=50
rankings=[]
for row in dev:
    q=vectorizer.transform([row["claim"]])
    scores=np.asarray(doc_matrix @ q.T).ravel()
    # Stable tie-breaking by original corpus index
    order=np.lexsort((np.arange(len(scores)), -scores))[:RESERVE_K]
    rankings.append({
        "claim_id": int(row["id"]),
        "doc_ids": doc_ids[order].astype(int).tolist(),
        "scores": scores[order].astype(float).tolist()
    })
print("Claims ranked:", len(rankings))
print("Example top documents:", rankings[0]["doc_ids"][:TOP_K])


In [ ]:
# Save ranking in JSONL and a compact CSV for audit
out=BASE/"results"; out.mkdir(exist_ok=True)
with open(out/"clean_rankings.jsonl","w",encoding="utf-8") as f:
    for r in rankings: f.write(json.dumps(r)+"\n")
pd.DataFrame([{"claim_id":r["claim_id"],"rank":j+1,"doc_id":d,"score":s}
              for r in rankings for j,(d,s) in enumerate(zip(r["doc_ids"],r["scores"]))]).to_csv(out/"clean_rankings.csv",index=False)
print("Saved clean rankings.")


## Baseline interpretation
This is the official TF-IDF *abstract retrieval component* adapted to produce a reserve list. It is not the official end-to-end SciFact verifier. The reserve list is retained to make the later corruption/recovery intervention reproducible.